# UK public contracts: the newest notices from Find a Tender and Contracts Finder

Runs [UK Public Contracts Lookup](https://apify.com/yappman/contracts-lookup) with its
default input: a search over the last 31 days at Find a Tender and Contracts Finder,
newest first, capped at 20 notices. Each row is one government procurement notice in one
fixed schema, with the buyer, the estimated value, the stage and the deadline, and designed not
to include individuals' details. The notebook shows the rows as a table and then reads the run summary,
which says whether either service left days uncovered.

**Cost at Apify Free-tier prices:** at most **$0.105** per run (20 notices at $0.005,
plus $0.005 for the run). The price is the same on every plan. `maxItems` caps the rows
and `max_total_charge_usd` caps the bill.

More data: [data.yappman.com](https://data.yappman.com).

In [ ]:
import os
from datetime import timedelta
from decimal import Decimal
import time

import pandas as pd
from apify_client import ApifyClient

TOKEN = os.environ.get("APIFY_TOKEN")
if not TOKEN:
    raise RuntimeError("Set the APIFY_TOKEN environment variable first. See the README.")
client = ApifyClient(TOKEN)
pd.set_option("display.html.use_mathjax", False)  # show third-party text as text


def run_actor(actor_id: str, run_input: dict, max_charge_usd: float):
    """Start an Actor, wait for it, and return the finished run.

    Spend is capped by the size of the input and by max_total_charge_usd, which
    stops the run if its charges would pass the cap. The run times out after 10 minutes.
    If a run is not SUCCEEDED, check it in the Console before running the cell again.
    """
    run = client.actor(actor_id).call(
        run_input=run_input,
        max_total_charge_usd=Decimal(str(max_charge_usd)),
        run_timeout=timedelta(minutes=10),
    )
    if run is None or run.status != "SUCCEEDED":
        status = None if run is None else run.status
        where = "" if run is None else f" Run: https://console.apify.com/actors/runs/{run.id}"
        raise RuntimeError(f"{actor_id} did not succeed: {status}.{where}")
    print(f"{actor_id}: run {run.id}")
    # Charges settle a few seconds after the run ends, so read the record again before printing them.
    time.sleep(5)
    settled = client.run(run.id).get()
    print("Charged events:", settled.charged_event_counts)
    return run

## 1. Run the search

The input is the Actor's default: search mode, both default services, 20 notices.
Public Contracts Scotland is opt-in and is not read here.

In [ ]:
ACTOR = "yappman/contracts-lookup"
RUN_INPUT = {
    "mode": "search",
    "sources": ["fts", "cf"],
    "maxItems": 20,
}

# Free tier: 20 notices x $0.005 + 1 run x $0.005 = $0.105
run = run_actor(ACTOR, RUN_INPUT, max_charge_usd=0.11)
rows = list(client.dataset(run.default_dataset_id).iterate_items())
print(len(rows), "notices")

## 2. The notices

Rows come out newest `publishedAt` first. A missing value is `null`. A procurement
published on both services is two rows; `reconcileKey` is a hash of the normalised buyer
and title that probably matches them, so join on it yourself.

In [ ]:
df = pd.DataFrame(rows)
if df.empty:
    print("No notices returned.")
else:
    df = df.reindex(columns=["publishedAt", "source", "title", "buyer", "value", "currency",
                             "stage", "deadline", "regions"])
    df["regions"] = df["regions"].apply(lambda r: ", ".join(r) if isinstance(r, list) else None)
    display(df)

## 3. The run summary

Find a Tender is currently read from its API only, so a wide search can come back
partial. The summary names any days that were not covered; a search that is complete
shows none.

In [ ]:
record = client.key_value_store(run.default_key_value_store_id).get_record("RUN_SUMMARY")
summary = (record or {}).get("value") or {}
print({k: summary.get(k) for k in ("status", "pushed", "readData", "runCharged")})
for source, info in (summary.get("sources") or {}).items():
    info = info or {}
    print(source, {k: info.get(k) for k in ("noticesRead", "missingDays", "staleSince") if k in info})

## Notes

* Contains public sector information licensed under the Open Government Licence v3.0.
  Keep this attribution with any data you republish.
* Rows hold no contact names, emails, phone numbers, addresses or supplier names. Titles
  are redacted by pattern, so a name written some other way in a title is not caught.
* Rows belong to the Actor's data terms, not to this repository's MIT licence.